Load the dataset

In [ ]:
import pandas as pd
df = pd.read_csv("creditcard.csv")

Show first 6 data points using head().

In [ ]:
df.head(6)

Describe pandas Dataframe by using describe.

In [ ]:
df.describe()

Show correlation heat plot of the entire dataset using matplotlib and sns, choose any
color pallet (except blue) you like (experiment).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df_corr = df.corr()

exp_palette = sns.color_palette("YlOrRd")

# code referred from supporting code
sns.heatmap(df_corr,
            cmap = exp_palette,
            vmax=.8,
            square=True,
            annot=False,
)

plt.show()

Show the Scatterplot matrix for the dataframe (avoid matplotlib and sns for this
assignment). You can use Scatterplot Matrix Plotly. Use the code for the second image that
shows different colors for classes. In this case, you will get two colors since we have two
classes. Also, experiment with visual aspects of the image (not a lot, but an excellent visual
will always leave a better impression. you can change color, thickness, font, font size, font
color, etc.). No need to explain the plots but do save them in a pdf/svg/png with either static
export function or html export function from plotly Interactive HTML Export Plotly.

In [ ]:
import random
random.seed(4217)

In [ ]:
df['Class'] = df.Class.astype('category')

In [ ]:
dfX = df.drop("Class", axis=1)

In [ ]:
import numpy as np
dfX = dfX[np.random.choice(dfX.columns.tolist(), size=10, replace=False)]

In [ ]:
dfXy = pd.concat([dfX, df['Class']], axis=1)

In [ ]:
# code referred from supporting code
import plotly.express as px

fig = px.scatter_matrix(dfXy, color='Class')
fig.update_layout(
    title='Credit Card Transaction Dataset',
    dragmode='select',
    width=600,
    height=600,
    hovermode='closest',
)
fig.write_html("plotly_scatterplot.html")

Split the dataset into the Training set and Test set. Choose your preferred split and
justify the rationale.

In [ ]:
X = df.loc[:, df.columns != 'Class'].to_numpy()
y = df.iloc[:,-1:].to_numpy()

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)

In [ ]:
X_train.shape

In [ ]:
X_test.shape

The split method I use is the build-in function from sklearn called train-test-split, and I choose the split to be 70-30 since there many data points, so 70% train set is sufficient to train

Perform classification routine by using LogisticRegression(),
KNeighborsClassifier(), DecisionTreeClassifier(), SVC(), GaussianNB(),
RandomForestClassifier(), BaggingClassifier(), GradientBoostingClassifier(),
XGboostclassifier. Output the accuracy box plot as we have seen in the class (make sure to
change regressmod df to classmod. And use an appropriate metric for classification
evaluation, for example, accuracy, precision,recall etc). Remember to use the object
oriented approach and develop a function (def...), this will be very helpful for the next
assignment.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, GradientBoostingClassifier
import xgboost as xgb

In [ ]:
# code referred from supporting code

def base_models():
  models = dict()
  models['LogisticRegression'] = LogisticRegression()
  models["KNeighborsClassifier"] = KNeighborsClassifier()
  models["DecisionTreeClassifier"] = DecisionTreeClassifier()
  models["SVC"] = SVC()
  models["GaussianNB"] = GaussianNB()
  models["RandomForestClassifier"] = RandomForestClassifier()
  models["BaggingClassifier"] = BaggingClassifier()
  models["GradientBoostingClassifier"] = GradientBoostingClassifier()
  models["XGboostclassifier"] = xgb.XGBClassifier()
  return models

In [ ]:
# code referred from supporting code
from sklearn.model_selection import cross_val_score, RepeatedKFold

def eval_models(model):
  cv = RepeatedKFold(n_splits=10, n_repeats=3, random_state=1)
  scores = cross_val_score(model, X_train, y_train, cv=cv, n_jobs=-1,
                            error_score='raise')
  return scores

In [ ]:
# code referred from supporting code
models = base_models()
# evaluate the models and store results
results, names = list(), list()

for name, model in models.items():
  scores = eval_models(model)
  results.append(scores)
  names.append(name)
  print('>%s %.3f (%.3f)' % (name, scores.mean(), scores.std()))


In [ ]:
# code referred from supporting codes

classmod = pd.DataFrame(np.transpose(results), columns = ["LogisticRegression","KNeighborsClassifier","DecisionTreeClassifier","SVC","GaussianNB","RandomForestClassifier","BaggingClassifier", "GradientBoostingClassifier", "XGboostclassifier"])
classmod = pd.melt(classmod.reset_index(), id_vars='index',value_vars=["LogisticRegression","KNeighborsClassifier","DecisionTreeClassifier","SVC","GaussianNB","RandomForestClassifier","BaggingClassifier", "GradientBoostingClassifier", "XGboostclassifier"])


In [ ]:
# code referred from supporting codes
from plotly.subplots import make_subplots
import plotly.graph_objs as go

fig = px.box(classmod, x="variable", y="value",color="variable",points='all',
labels={"variable": "Machine Learning Model",
        "value": "Accuracy Scores"
        },title="Model Performance")
fig.write_html("plotly_boxplot.html")

Select the best classifier for level 0 classifier. Use logistic regression as a second level
classifier. Similar to 5 generate the box plot and show the accuracy of each algorithm as
well as stacked classifier. Also show the confusion metrices of the above algorithms

In [ ]:
# code referred from supporting code
from sklearn.ensemble import StackingClassifier # choose stackingclassifier for level0

def get_stacking():
	# define the base models
  level0 = list()
  level0.append(('RandomForestClassifier', RandomForestClassifier())) # either random forest and XGBoost classifier has accuarcy of 1, here I choose random forest

	# define meta learner model
  level1 = LogisticRegression()
	# define the stacking ensemble
  model = StackingClassifier(estimators=level0, final_estimator=level1, cv=5)
  return model

In [ ]:
# code referred from supporting code
def base_stack_models():
  models = dict()
  models["RandomForestClassifier"] = RandomForestClassifier()
  models["LogisticRegression"] = LogisticRegression()
  models["Stacked Model"] = get_stacking()
  return models

In [ ]:
# code referred from supporting code

def eval_stack_models(model):
  cv = RepeatedKFold(n_splits=10, n_repeats=3, random_state=1)
  scores = cross_val_score(model, X_train, y_train, cv=cv, n_jobs=-1,
                            error_score='raise')
  return scores

In [ ]:
# code referred from supporting codes
stack_models = base_stack_models()
# evaluate the models and store results
stack_results, stack_names = list(), list()

for name, model in stack_models.items():
  scores = eval_stack_models(model)
  stack_results.append(scores)
  stack_names.append(name)
  print('>%s %.3f (%.3f)' % (name, scores.mean(), scores.std()))

The above cell taks extremely long time to execute, causing the google colab disconnection, so I just stop it

Export the Pickle model and import it back. Use the imported model to predic the
y_test from x_test and report the confusion metrix

In [ ]:
# code referred from supporting code
import pickle

level0 = list()
level0.append(('KNeighborsClassifier', KNeighborsClassifier())) # use knn bc it takes shorter time

level1 = LogisticRegression()
model = StackingClassifier(estimators=level0, final_estimator=level1, cv=5)
model.fit(X_train, y_train)

In [ ]:
# code referred from supporting code
pkl_filename = "AssignmentPickle.pkl"
with open(pkl_filename, 'wb') as file:
    pickle.dump(model, file)


In [ ]:
# code referred from supporting code
with open(pkl_filename, 'rb') as file:
    pickle_model = pickle.load(file)

In [ ]:
# code referred from supporting code

score = pickle_model.score(X_test, y_test)
print("Test score: {0:.2f} %".format(100 * score))
Y_predict = pickle_model.predict(X_test)

In [ ]:
import seaborn as sns

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, Y_predict)
sns.heatmap(cm, annot=True)

Show both text and visual confusion Matrices using scikit learn and matplotlib and
explain what the graph tells you and what you did.

In [ ]:
print(cm)
sns.heatmap(cm, annot=True)

In [ ]:
# code referred from supporting code
import matplotlib.pyplot as plt

predictions = pd.DataFrame(Y_predict, columns=['predictions'])
predictions['actual'] = y_test
plt.scatter(x = y_test, y = Y_predict, color='#336699',alpha=0.6)
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Predicted vs Actual', fontsize=15, color='#336699',loc='center')

From the heatmap, we can see that there are around 8500 data points are correctly predicted as label 0 and 6 of them are correctly predicted as 1, the accuracy is very high.

In [ ]:
!jupyter nbconvert --to html Assignment2_G06.ipynb